In [12]:
import ijson
import json
from decimal import Decimal
from pathlib import Path

In [ ]:
BASE_DIR = Path().resolve().parent.parent

DATA_DIR = BASE_DIR / "data" / "processed"
PROD_IN  = DATA_DIR / "A_research_products.json"        # input
PROD_OUT = DATA_DIR / "A_research_products_final.json"  # output

In [ ]:
def to_serializable(obj):
    if isinstance(obj, Decimal):
        return float(obj)
    if isinstance(obj, dict):
        return {k: to_serializable(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_serializable(v) for v in obj]
    return obj

def analyze_null_percentages(input_path, label=""):
    null_counts = {}
    total_count = 0
    
    with open(input_path, "r", encoding="utf-8") as f:
        for product in ijson.items(f, "item"):
            total_count += 1
            for key, value in product.items():
                null_counts[key] = null_counts.get(key, 0) + (1 if value is None else 0)
    
    print(f"Null value percentage per key ({label}):\n")
    print(f"{'Key':30} {'Null %':>8}")
    print("-" * 40)
    
    null_percentages = {}
    for key, count in null_counts.items():
        percent = (count / total_count) * 100 if total_count > 0 else 0
        null_percentages[key] = percent
        print(f"{key:30} {percent:8.2f}")
    
    return null_percentages

def filter_keys_by_null_percentage(input_path, output_path, null_percentages, threshold):
    """
    Filters keys from research products based on null percentage and removes unwanted keys.
    Only keeps products of type 'publication'. Also cleans up author fields and extracts keywords, SDG, FOS.
    """
    # Select keys to keep based on null percentage and explicit removal list
    keys_to_keep = [k for k, p in null_percentages.items() if p <= threshold]
    keys_to_remove = {
        "language", "collectedFrom", "descriptions", "bestAccessRight", "countries", "subjects",
        "publisher", "isGreen", "isInDiamondJournal", "affiliated_organizations", "organizations",
        "sources", "publiclyFunded", "originalIds", "instances", "pids", "container"
    }
    keys_to_keep = [k for k in keys_to_keep if k not in keys_to_remove]

    with open(input_path, "r", encoding="utf-8") as f_in, open(output_path, "w", encoding="utf-8") as f_out:
        f_out.write("[\n")
        first = True

        for product in ijson.items(f_in, "item", use_float=True):
            if product.get("type") != "publication":
                continue

            # Filter keys
            filtered = {k: product[k] for k in keys_to_keep if k in product}

            # Clean author fields
            authors = filtered.get("authors")
            if isinstance(authors, list):
                for author in authors:
                    for k in ["name", "surname", "provenance"]:
                        author.pop(k, None)
                    if isinstance(author.get("pid"), dict):
                        author["pid"].pop("provenance", None)

            # Extract keywords, SDG, FOS from subjects
            keywords, sdg, fos = [], [], []
            subjects = product.get("subjects", [])
            if isinstance(subjects, list):
                for subj in subjects:
                    subject_info = subj.get("subject")
                    if subject_info:
                        scheme = subject_info.get("scheme")
                        value = subject_info.get("value")
                        if scheme == "keyword" and value:
                            keywords.append(value)
                        elif scheme == "SDG" and value:
                            sdg.append(value)
                        elif scheme == "FOS" and value:
                            fos.append(value)

            filtered["keywords"] = "; ".join(keywords)
            filtered["SDG"] = "; ".join(sdg)
            filtered["FOS"] = "; ".join(fos)

            if not first:
                f_out.write(",\n")
            json.dump(filtered, f_out, ensure_ascii=False, indent=2)
            first = False

        f_out.write("\n]")

In [15]:
null_percentages = analyze_null_percentages(PROD_IN, "Complete")

Null value percentage per key (Complete):

Key                              Null %
----------------------------------------
authors                            0.13
openAccessColor                   78.10
publiclyFunded                     2.69
type                               0.00
language                           0.00
countries                         11.59
subjects                          33.92
mainTitle                          0.00
subTitle                         100.00
descriptions                      31.00
publicationDate                    0.04
publisher                         26.93
embargoEndDate                    95.04
sources                           47.27
formats                           71.51
contributors                      90.38
coverages                        100.00
bestAccessRight                    3.27
container                         49.66
documentationUrls                100.00
codeRepositoryUrl                100.00
programmingLanguage              100

In [16]:
filter_keys_by_null_percentage(PROD_IN, PROD_OUT, null_percentages, threshold=50.0)
analyze_null_percentages(PROD_OUT, "Filtered")

Null value percentage per key (Filtered):

Key                              Null %
----------------------------------------
authors                            0.13
type                               0.00
mainTitle                          0.00
publicationDate                    0.04
id                                 0.00
indicators                         0.04
keywords                           0.00
SDG                                0.00
FOS                                0.00


{'authors': 0.1332427433661722,
 'type': 0.0,
 'mainTitle': 0.0,
 'publicationDate': 0.03887819423998792,
 'id': 0.0,
 'indicators': 0.03887819423998792,
 'keywords': 0.0,
 'SDG': 0.0,
 'FOS': 0.0}